# 04 — Frontier-Vergleich (Phase 5)

Eure Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) gegen Frontier-LLM-Annotation derselben 12 Anzeigen (`annotation/frontier_gold.csv`). Output: κ-Tabelle, drei Disagreement-Beispiele, Material fürs Make-or-Buy-Memo (`memo_make_or_buy.md` im Repo-Root).

Cheatsheet: `CHEATSHEETS/frontier-llm-workflow.md`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _01.06.2026_ |
| Frontier-Modell | `claude-opus-4-8 - medium - no thinking`|
| Prompt-Variante | System-Prompt aus Iteration B mit 1 Few-Shot-Beispiel
| Anzahl Korrektur-Turns | 0 |
| Schema-Verletzungs-Mapping | keine (Werte, die gemappt wurden — z. B. *möglich → teilweise*) |
| Auffälligkeiten | Das Modell hat ein exakt valides JSON generiert. |
| Frontier-CSV | `annotation/frontier_gold.csv` |
| Eigene Gold-CSV | `annotation/meine_gold.csv` |

## Frontier-Daten laden + Schema-Konformität prüfen

In [4]:
import json
from pathlib import Path
import pandas as pd

# 1. Pfade definieren
basispfad = Path("/home/jovyan/work/notebooks/LLM-Workshop/llm-workshop")
gold_pfad = basispfad / "AUFGABEN" / "annotation" / "meine_gold.csv" 
frontier_pfad = basispfad / "AUFGABEN" / "annotation" / "frontier_gold.csv"

# 2. Daten laden
df_human = pd.read_csv(gold_pfad, encoding="utf-8-sig", dtype=str)
df_frontier = pd.read_csv(frontier_pfad, encoding="utf-8-sig", dtype=str)

# Spaltennamen säubern und vereinheitlichen
df_human.columns = df_human.columns.str.strip()
df_frontier.columns = df_frontier.columns.str.strip()

if "id" in df_human.columns:
    df_human = df_human.rename(columns={"id": "refnr"})

df_human["refnr"] = df_human["refnr"].astype(str).str.strip()
df_frontier["refnr"] = df_frontier["refnr"].astype(str).str.strip()

# 3. Datensätze zusammenführen
merged_eval = pd.merge(
    df_human, df_frontier, on="refnr", suffixes=("_human", "_frontier")
)
print(
    f"Erfolgreich gejoint! {len(merged_eval)} von 12 Anzeigen für den Kappa-Vergleich bereit.\n"
)

# Schneller Blick auf die geladenen Spalten von Claude
print("Spalten im Frontier-Datensatz:", df_frontier.columns.tolist())

Erfolgreich gejoint! 12 von 12 Anzeigen für den Kappa-Vergleich bereit.

Spalten im Frontier-Datensatz: ['refnr', 'titel', 'homeoffice', 'vertragsart', 'erfahrungslevel', 'gehalt_min_eur', 'gehalt_zeitraum', 'skills_top3']


## κ Mensch ↔ Frontier

Hypothese-Cell *vor* dem κ-Compute: was schätzt ihr ist das Gesamt-κ? Bei welchem Feld erwartet ihr das niedrigste κ — warum?

### Hypothese vor dem κ-Compute
* Gesamt-K wird durch das validate.py nicht angezeigt.
* **Niedrigstes Feld & Begründung:** Ich erwarte das niedrigste Kappa beim Feld `Erfahrungslevel`. Diese Kategorie lässt sich sowohl für Mensch als auch für Maschine nicht eindeutig benennen, da dazu einheitliche Standards fehlen oder sie nicht klar ausgeschrieben werden.

In [1]:
# Validierungsskript mit absoluten Pfaden direkt aus dem Notebook heraus starten
!python /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/AUFGABEN/annotation/validate.py /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/AUFGABEN/annotation/meine_gold.csv --kappa-against /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/AUFGABEN/annotation/frontier_gold.csv

OK: meine_gold.csv ist schema-konform.

Cohen's kappa: meine_gold.csv vs. frontier_gold.csv
Gemeinsame IDs: 12

Feld                    kappa   Übereinst.
------------------------------------------
homeoffice              0.415         67%
vertragsart             1.000        100%
erfahrungslevel         0.318         58%

Interpretation (Landis & Koch 1977):
  < 0.00      schlechter als Zufall
  0.00–0.20   schlecht
  0.21–0.40   mäßig
  0.41–0.60   moderat
  0.61–0.80   substanziell
  0.81–1.00   fast perfekt


## Drei Disagreement-Beispiele

Pro Beispiel: Anzeige-ID, euer Wert, Frontier-Wert, **wer hatte recht** — Begründung mit Bezug auf Schema-Definition oder konkrete Anzeigen-Stelle. Material fürs Make-or-Buy-Memo.

- `15939-BB-632493-7878-9058-S` - Opus und partner_gold.csv haben Erfahrungslevel als "junior" gelabelt. In meine_gold.csv als "nicht_genannt" gelabelt, da auch hier wieder ein eindeutiger Standard für diese Bezeichnungen fehlen, insbesondere für solche Edge-Cases wie Praktikumsstellen.

- `15939-BB-633455-7878-6343-S` - Erfahrungslevel - Opus und partner_gold.csv haben es als "senior" eingeordnet, wahrscheinlich aufgrund der Nennung von mehrjähriger Berufserfahrung und einem abgeschlossenen Studium. In meine_gold.csv als "nicht_genannt" gelabelt, weil mehrjährige Berufserfahrung nicht eindeutig einem Erfahrungslevel aus meiner Sicht zuzuordnen ist. Mehrjährig kann auch unter 5 Jahre bedeuten - in manchen Branchen gilt das noch nicht als senior, manchmal reichen dort auch 3 Jahre aus.
 
- `13635-7fbe73ac_JB5131141-S` - Homeoffice - wir als "teilweise" gelabelt, Opus als "remote" gelabelt. In der Anzeige wird eindeutig "mobiles Arbeiten/Homeoffice" benannt, aber im Gesamtkontext wird vorher in der Stellenbeschreibung auch vom "Büroleben" gesprochen, daher aus menschlicher Sicht keine eindeutige und reine remote-Stelle, da durchaus auch die Möglichkeit zur Arbeit im Büro besteht.


Wer hatte Recht? - Stellenweise gibt es bei drei verschiedenen Betrachtern (2 x Mensch, 1 x KI-Modell) drei verschiedene Labels für eine Kategorie auf Grundlage der identischen Stellenanzeige. Besonders bei nicht eindeutig genannten Themenfeldern wie Homeoffice oder Erfahrungslevel ist diese Frage nicht eindeutig zu beantworten, aufgrund von fehlenden Standards bzw. unterschiedlichen Interpretationen, die je nach Auslegung alle richtig sein können. 